# Flan-T5 Unified — Fine-tuning Intent + Entity

Fine-tune `google/flan-t5-base` pour faire intent classification ET entity extraction en un seul modèle.

**Format de sortie** : `INTENTION: VOYAGE | DEPART: Paris | ARRIVEE: Lyon | VIA: Dijon`

**Durée estimée** : ~30-45min sur T4

In [ ]:
!pip install -q transformers datasets accelerate
print('Done')

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

model_name = 'google/flan-t5-base'
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f'Modele charge sur {device}')

In [ ]:
from google.colab import files
print('Uploadez train.json et val.json depuis datasets/base/')
uploaded = files.upload()

In [ ]:
import json
from datasets import Dataset

PROMPT = 'Analyse cette phrase de voyage: {text}'

INTENT_MAP = {
    'TRIP': 'VOYAGE',
    'NOT_TRIP': 'PAS_VOYAGE',
    'UNKNOWN': 'INCONNU',
    'NOT_FRENCH': 'PAS_FRANCAIS',
}

def format_example(example):
    text = example.get('sentence', '')
    intent = example.get('intent', 'UNKNOWN')
    intent_fr = INTENT_MAP.get(intent, 'INCONNU')
    
    dep = example.get('departure', '') or 'aucun'
    dest = example.get('destination', '') or 'aucun'
    inter = example.get('intermediate', '')
    if isinstance(inter, list):
        inter = ', '.join(inter) if inter else 'aucun'
    elif not inter:
        inter = 'aucun'
    
    target = f'INTENTION: {intent_fr} | DEPART: {dep} | ARRIVEE: {dest} | VIA: {inter}'
    
    return {
        'input_text': PROMPT.format(text=text),
        'target_text': target,
    }

with open('train.json', 'r') as f:
    train_data = json.load(f)
with open('val.json', 'r') as f:
    val_data = json.load(f)

# Limiter pour la vitesse (augmenter si vous avez le temps)
train_data = train_data[:10000]
val_data = val_data[:1000]

train_dataset = Dataset.from_list([format_example(ex) for ex in train_data])
val_dataset = Dataset.from_list([format_example(ex) for ex in val_data])

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)}')
print(f'\nExemple input:  {train_dataset[0]["input_text"]}')
print(f'Exemple output: {train_dataset[0]["target_text"]}')

In [ ]:
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples['input_text'],
        max_length=256,
        truncation=True,
        padding='max_length',
    )
    labels = tokenizer(
        examples['target_text'],
        max_length=128,
        truncation=True,
        padding='max_length',
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=train_dataset.column_names)
val_tokenized = val_dataset.map(tokenize_function, batched=True, remove_columns=val_dataset.column_names)

print('Tokenisation OK')

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir='flan-t5-unified',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    fp16=True,
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    predict_with_generate=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print('Lancement...')
trainer.train()

In [ ]:
model.eval()

test_cases = [
    'Je veux aller de Paris a Lyon',
    'Je veux aller de Marseille a Nice en passant par Toulon',
    'Quel temps fait-il demain ?',
    'Hello how are you?',
    'Un billet pour Bordeaux svp',
]

print('=' * 70)
for text in test_cases:
    prompt = PROMPT.format(text=text)
    inputs = tokenizer(prompt, return_tensors='pt', max_length=256, truncation=True).to(device)
    outputs = model.generate(**inputs, max_new_tokens=128)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f'Input:  {text}')
    print(f'Output: {result}')
    print('-' * 70)

In [ ]:
model.save_pretrained('flan-t5-unified')
tokenizer.save_pretrained('flan-t5-unified')
print('Modele sauvegarde')

!zip -r flan-t5-unified.zip flan-t5-unified/

from google.colab import files
files.download('flan-t5-unified.zip')
print('Place le contenu du zip dans models/flan-t5-travel/ du projet (remplace lancien)')